# SCPCP000004 — clean single-cell analysis

Cleaned from [your original notebook](https://colab.research.google.com/drive/1h1fKZPZhQ5j-l8qegFGt1Er8rh2Zd_Ow).
Keeps scVI integration, Leiden clustering, UMAPs, reference/patient summaries, positive markers,
the original 24-gene dot plot, and quantitative panel tables. All code is included here.

Run top to bottom. Existing clustered/processed AnnData and scVI models are reused by default.
An A100/high-RAM runtime is needed if scVI must be trained; the dataset is large even when
only regenerating plots. All exports go into `single_cell_analysis_clean_v1/` on Drive.

**Scope:** this preserves the original exploratory whole-cohort analysis. It does not replace
the donor-separated 122 benchmark cases, which were generated independently within samples.
These merged clusters/markers and public reference labels must not become formal model
training/test inputs. Existing raw data, old results and benchmark cases are unchanged.

The original installed versions and analysis settings are retained. New scVI fits use seed 0;
saved original embeddings/labels are preferred, since a new fit need not reproduce them exactly.


## 1. Drive and settings


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json

import os
ROOT = Path(os.environ.get('GTC_DRIVE_ROOT', '/content/drive/MyDrive/GTC2027/SCPCP000004'))
OUT=ROOT/'single_cell_analysis_clean_v1'
OUT.mkdir(exist_ok=True)
FORCE_RECOMPUTE=False
SAVE_H5AD=True
RAW=ROOT/'SCPCP000004_merged_rna.h5ad'
sources=[OUT/'SCPCP000004_analysis.h5ad',
         ROOT/'SCPCP000004_scvi_clustered_rna.h5ad',
         ROOT/'SCPCP000004_scvi_processed_rna.h5ad',RAW]
INPUT=next((p for p in sources if p.exists()),None)
assert INPUT is not None, 'Place the original merged or processed .h5ad file in ROOT.'
print('Input:',INPUT)
print('Exports:',OUT)


## 2. Install the analysis dependencies


In [ ]:
import subprocess,sys
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'scanpy==1.12.5','scvi-tools==1.5.2','igraph==1.0.0','leidenalg==0.12.0'])
import scanpy as sc
import scvi
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy import sparse
from IPython.display import display
import importlib.metadata as metadata
scvi.settings.seed=0
torch.set_float32_matmul_precision('medium')


## 3. Load the data


In [ ]:
adata=sc.read_h5ad(INPUT)
for key in ['sample_id','participant_id','openscpca_celltype_annotation']:
    assert key in adata.obs, f'Missing observation field: {key}'
assert 'gene_symbol' in adata.var
assert adata.obs_names.is_unique and adata.var_names.is_unique
assert adata.uns.get('X_name')=='logcounts', 'Verify the expression matrix before marker analysis.'
print(adata)
print('Samples:',adata.obs.sample_id.nunique(),'Participants:',adata.obs.participant_id.nunique())


## 4. Reuse or fit scVI
Preserves the original count layer (`spliced`) and batch key (`sample_id`). This section skips fitting when `X_scVI` is already saved.


In [ ]:
embedding_changed=FORCE_RECOMPUTE or 'X_scVI' not in adata.obsm
if embedding_changed:
    assert torch.cuda.is_available(), 'Select an A100 GPU before fitting scVI.'
    assert 'spliced' in adata.layers
    adata.layers['spliced']=sparse.csr_matrix(adata.layers['spliced'])
    values=adata.layers['spliced'].data
    for start in range(0,len(values),1000000):
        block=values[start:start+1000000]
        assert np.isfinite(block).all() and (block>=0).all()
        assert np.allclose(block,np.rint(block),rtol=0,atol=1e-6), 'spliced must contain counts.'
    scvi.model.SCVI.setup_anndata(adata,layer='spliced',batch_key='sample_id')
    saved=next((p for p in [OUT/'scvi_model',ROOT/'scvi_model'] if (p/'model.pt').is_file()),None)
    if saved is not None and not FORCE_RECOMPUTE:
        scvi_model=scvi.model.SCVI.load(str(saved),adata=adata)
        print('Loaded saved scVI model:',saved)
    else:
        scvi_model=scvi.model.SCVI(adata)
        scvi_model.train(accelerator='gpu',devices=1)
        scvi_model.save(str(OUT/'scvi_model'),overwrite=FORCE_RECOMPUTE)
    adata.obsm['X_scVI']=scvi_model.get_latent_representation()
else:
    print('Reusing saved X_scVI; no scVI training.')


## 5. Leiden clustering and UMAP


In [ ]:
params=adata.uns.get('neighbors',{}).get('params',{})
clusters_changed=bool(embedding_changed or 'leiden_scvi' not in adata.obs
    or 'X_umap' not in adata.obsm or params.get('use_rep')!='X_scVI'
    or params.get('n_neighbors')!=20)
if clusters_changed:
    sc.pp.neighbors(adata,use_rep='X_scVI',n_neighbors=20,random_state=0)
    sc.tl.leiden(adata,resolution=.5,key_added='leiden_scvi',flavor='igraph',
                 directed=False,n_iterations=2,random_state=0)
    sc.tl.umap(adata,random_state=0)
else:
    print('Reusing saved clusters and UMAP.')
print('Clusters:',adata.obs.leiden_scvi.nunique())
fig=sc.pl.umap(adata,color=['leiden_scvi','openscpca_celltype_annotation'],
               wspace=.5,show=False,return_fig=True)
fig.savefig(OUT/'clusters_and_reference_umap.png',dpi=180,bbox_inches='tight')
display(fig);plt.close(fig)
fig=sc.pl.umap(adata,color='sample_id',show=False,return_fig=True,title='Samples in the scVI embedding')
fig.savefig(OUT/'sample_umap.png',dpi=180,bbox_inches='tight')
display(fig);plt.close(fig)


## 6. Cluster and patient summaries
Public annotations provide comparison context, not independent expert ground truth.


In [ ]:
def cluster_tables(obs):
    import pandas as pd
    groups=obs['leiden_scvi']
    refs=pd.crosstab(groups,obs['openscpca_celltype_annotation'].astype('string').fillna('Missing'))
    patients=pd.crosstab(groups,obs['participant_id'].astype('string').fillna('Missing'))
    known=patients.drop(columns='Missing',errors='ignore')
    summary=pd.DataFrame({'cells':refs.sum(axis=1),
        'dominant_reference':refs.idxmax(axis=1),
        'dominant_reference_fraction':refs.max(axis=1)/refs.sum(axis=1),
        'participants':known.gt(0).sum(axis=1),
        'largest_patient_fraction':known.max(axis=1)/patients.sum(axis=1)})
    return summary,refs,patients


cluster_summary,reference_counts,patient_counts=cluster_tables(adata.obs)
cluster_summary.to_csv(OUT/'cluster_summary.csv')
reference_counts.to_csv(OUT/'cluster_reference_counts.csv')
patient_counts.to_csv(OUT/'cluster_patient_counts.csv')
display(cluster_summary)


## 7. Positive marker genes
Uses the original log-expression matrix and Wilcoxon comparison. Fold changes are omitted because the historical log base has not been established. Missing/blank gene symbols fall back to gene IDs.


In [ ]:
def marker_table(frame):
    frame=frame.loc[frame['scores'].gt(0)].copy()
    symbol=frame['gene_symbol'].astype('string')
    frame['gene']=symbol.mask(symbol.str.strip().eq('')).fillna(frame['names'].astype('string'))
    return frame.drop(columns='logfoldchanges',errors='ignore')


marker_params=adata.uns.get('cluster_markers',{}).get('params',{})
if clusters_changed or marker_params.get('groupby')!='leiden_scvi' or marker_params.get('method')!='wilcoxon':
    sc.tl.rank_genes_groups(adata,groupby='leiden_scvi',method='wilcoxon',
        reference='rest',use_raw=False,n_genes=30,pts=True,key_added='cluster_markers')
else:
    print('Reusing saved Wilcoxon marker results.')
markers=marker_table(sc.get.rank_genes_groups_df(adata,group=None,key='cluster_markers',gene_symbols='gene_symbol'))
markers.to_csv(OUT/'SCPCP000004_cluster_markers.csv',index=False)
top10=markers.groupby('group',observed=True)['gene'].apply(lambda genes:', '.join(genes.head(10))).to_frame('top_10_markers')
top10.to_csv(OUT/'top_10_markers.csv')
display(top10)


## 8. Original marker panel and quantitative tables
Dot color is scaled within each gene; dot size is the fraction expressing it. Missing/ambiguous genes are listed explicitly. Mean values retain the source log-expression units.


In [ ]:
PANEL='CHGA CHGB PHOX2B TH DBH CD3D CD3E TRAC LYZ AIF1 CD68 COL1A1 DCN ACTA2 PDGFRB CDH19 SOX10 PLP1 PECAM1 VWF HBB HBA1 S100A8 S100A9'.split()
symbols=adata.var.gene_symbol.astype('string')
present=[gene for gene in PANEL if symbols.eq(gene).sum()==1]
missing=[{'gene':gene,'reason':'missing' if symbols.eq(gene).sum()==0 else 'ambiguous_duplicate_symbol'}
         for gene in PANEL if symbols.eq(gene).sum()!=1]
pd.DataFrame(missing,columns=['gene','reason']).to_csv(OUT/'unavailable_panel_genes.csv',index=False)
assert present, 'No unique panel genes found.'
dot=sc.pl.dotplot(adata,var_names=present,gene_symbols='gene_symbol',groupby='leiden_scvi',
    use_raw=False,standard_scale='var',figsize=(14,7),return_fig=True,show=False)
dot.savefig(OUT/'marker_dotplot.png',dpi=180)
dot.savefig(OUT/'marker_dotplot.pdf')
display(dot.fig);plt.close(dot.fig)
expression=sc.get.obs_df(adata,keys=present,gene_symbols='gene_symbol',use_raw=False)
groups=adata.obs['leiden_scvi']
expression.groupby(groups,observed=True).mean().to_csv(OUT/'cluster_panel_mean_log_expression.csv')
(expression>0).groupby(groups,observed=True).mean().to_csv(OUT/'cluster_panel_fraction_expressing.csv')
print('Saved panel tables and dot plots.')


## 9. Save the analysis and provenance


In [ ]:
manifest={'scope':'Exploratory whole-cohort analysis, not donor-separated benchmark cases',
    'original_notebook':'https://colab.research.google.com/drive/1h1fKZPZhQ5j-l8qegFGt1Er8rh2Zd_Ow',
    'input_path':str(INPUT),'input_bytes':INPUT.stat().st_size,
    'cells':int(adata.n_obs),'genes':int(adata.n_vars),'clusters':int(adata.obs.leiden_scvi.nunique()),
    'scvi_count_layer':'spliced','scvi_batch_key':'sample_id','new_fit_seed':0,
    'embedding_recomputed_or_loaded_from_model':bool(embedding_changed),'clusters_recomputed':bool(clusters_changed),
    'n_neighbors':20,'resolution':.5,'leiden_iterations':2,'random_state':0,
    'marker_method':'wilcoxon','marker_matrix':'X (source logcounts; historical log base unresolved)',
    'panel':PANEL,'environment':{p:metadata.version(p) for p in ['scanpy','scvi-tools','anndata','pandas','numpy','torch','igraph','leidenalg']}}
if SAVE_H5AD:
    adata.uns['clean_analysis_provenance']=json.dumps(manifest)
    adata.write_h5ad(OUT/'SCPCP000004_analysis.h5ad')
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2))
print('Analysis saved:',OUT)
